# Stage-1 L4 Consolidation Pipeline — rev.4
**역할 분담**: 🔵 노트북(결정론적 계산 전부) · 🟢 Claude(에이전트 판단 → 결정 JSON) · 🟠 교수(셀 실행 + 승인 2회)

| 셀 | 담당 | 내용 |
|---|---|---|
| S0–S2 | 🔵 | 설정 · 원본 로드 · 보호용어/수준 태깅 |
| **GATE-1** | 🟢→🟠 | 비리스크 판정(Claude) → 검토·승인(교수) |
| S3 | 🔵 | 승인 제거 적용 |
| **GATE-2** | 🟢 | 정규화 결정(형식만·기제 동결) |
| S4 | 🔵 | 정규화 적용 + 보호용어 위반 자동검사 |
| S5 | 🔵 | 임베딩 재생성 |
| **GATE-3** | 🟢 | 통합 사전(고유사 쌍 2인 판정) |
| S6 | 🔵 | 사전 선적용 + 제약 덴드로그램 + τ 절단 |
| **GATE-4** | 🟢 | 명명 검정(A→B 2인 + 수준 위반 기각) |
| S7 | 🔵 | 결정 적용 → C80/C70 확정 |
| S8 | 🔵 | 검증 + 산출물 + Before/After HTML |

GATE 셀은 결정 파일이 없으면 안내만 출력합니다. Claude가 파일을 만들면 다음 셀로 진행하세요.

In [8]:
# S0 — 설정
import json, os, re, hashlib, platform, datetime, csv
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, scipy
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

ROOT=Path.cwd()
while not (ROOT/'data/experiments/tau_tiers_v2_19/master_cards.json').exists():
    if ROOT.parent==ROOT: raise FileNotFoundError('repo root 못 찾음')
    ROOT=ROOT.parent
os.chdir(ROOT); print('repo root:',ROOT)

EXP=ROOT/'data/experiments/stage1'; EXP.mkdir(parents=True,exist_ok=True)
DEC=EXP/'decisions'; DEC.mkdir(exist_ok=True)
OUT=EXP/'out'; OUT.mkdir(exist_ok=True)
SRC=ROOT/'data/experiments/tau_tiers_v2_19/master_cards.json'
TD=ROOT/'reports/consolidation/v2_19_tier_design'

_m=ROOT/'tmp/models/bge-m3'
if (_m/'modules.json').exists():
    os.environ['HF_HUB_OFFLINE']='1'; MODEL=str(_m); print('local model OK')
else:
    MODEL='BAAI/bge-m3'; print('경고: 로컬 모델 없음')

CFG=dict(model=MODEL, max_seq_length=256, batch_size=32, linkage='complete',
  taus=dict(C80=0.80, C70=0.70), eps=0.001, block_cross_level=True,
  cannot_link=[('RAI4-0228','RAI4-0229'),('RAI4-0659','RAI4-0892'),('RAI4-0863','RAI4-0870'),
               ('RAI4-0682','RAI4-0691'),('RAI4-0435','RAI4-0569'),('RAI4-1275','RAI4-1307'),
               ('RAI4-1591','RAI4-1714')])
def sha(p): return hashlib.sha256(open(p,'rb').read()).hexdigest()[:16]
print('DEC(Claude):',DEC); print('OUT(노트북):',OUT)

repo root: /Users/deep1003/data3/RAI-Risk-Taxonomy
local model OK
DEC(Claude): /Users/deep1003/data3/RAI-Risk-Taxonomy/data/experiments/stage1/decisions
OUT(노트북): /Users/deep1003/data3/RAI-Risk-Taxonomy/data/experiments/stage1/out


In [9]:
# S1 — 원본 로드
doc=json.load(open(SRC))
cards=[dict(c) for c in doc['cards'] if c['status']=='active']
by={c['l4_id']:c for c in cards}
print('원본 활성 카드:',len(cards),'| src sha',sha(SRC))

원본 활성 카드: 1652 | src sha 9c059569826869ab


In [10]:
# S2 — 보호 용어 목록 + 수준(construct/instance) 태깅  [재작성 없음, 메타데이터만]
term=json.load(open(TD/'ko_term_dictionary.json'))
OFFICIAL={t['ko'].split('(')[0].strip() for t in term['terms']}
LITERATURE={'자동화 편향','목표 오정렬','통제 상실','권력 추구','환각','편향','견고성','설명 가능성',
 '가치 정렬','보상 해킹','명세 게이밍','기만','오용','인간 감독','적대적 공격','데이터 포이즈닝',
 '프롬프트 인젝션','탈옥','딥페이크','허위정보','개인정보 유출','목표의 잘못된 일반화','창발','책임 격차'}
GENERIC={'위험','안전','안전성','신뢰성','투명성','책임성','공정성','오용','기만','창발',
         '견고성','설명 가능성','편향','위험한 능력','인간 감독','가치 정렬',
         '고영향 인공지능','생성형 인공지능','개인정보 유출','허위정보','잘못된 정보'}
PROTECTED=sorted((OFFICIAL|LITERATURE)-GENERIC)  # 일반 명사 제외 (라벨 어디에나 나타나 오탐)

CTX=re.compile(r'(medical|clinical|health|driv|vehicle|hiring|recruit|credit|loan|court|judicial'
 r'|polic|welfare|educat|student|child|patient|worker|employee|military|weapon|financ|insur|border'
 r'|surveillance|robot|chatbot|search engine|recommend|moderation|consumer'
 r'|의료|임상|진료|운전|차량|채용|신용|대출|법원|사법|치안|복지|교육|학생|아동|환자|노동자|군사|무기'
 r'|금융|보험|국경|감시|로봇|자율주행|챗봇|추천|검열|소비자)', re.I)

def is_protected(c): return any(p in c['label_ko'] for p in PROTECTED)
def level(c):
    d=(c.get('definition_en') or '')+' '+(c.get('definition_ko') or '')
    return 'instance' if CTX.search(d) else 'construct'

for c in cards:
    c['level']=level(c); c['protected_term']=is_protected(c)
BASE_LEVEL={c['l4_id']:c['level'] for c in cards}
print('수준:',dict(Counter(c['level'] for c in cards)),
      '| 보호용어 카드:',sum(1 for c in cards if c['protected_term']),
      '| 보호용어 항목:',len(PROTECTED))
json.dump(PROTECTED, open(OUT/'protected_terms.json','w'), ensure_ascii=False, indent=1)
json.dump([{'l4_id':c['l4_id'],'label_ko':c['label_ko'],'label_en':c['label_en'],
            'definition_ko':c.get('definition_ko'),'definition_en':c.get('definition_en'),
            'level':c['level'],'protected':c['protected_term']} for c in cards],
          open(DEC/'_input_all_cards.json','w'), ensure_ascii=False, indent=0)
print('-> 🟢 Claude 입력:', DEC/'_input_all_cards.json')

수준: {'construct': 1218, 'instance': 434} | 보호용어 카드: 75 | 보호용어 항목: 25
-> 🟢 Claude 입력: /Users/deep1003/data3/RAI-Risk-Taxonomy/data/experiments/stage1/decisions/_input_all_cards.json


In [11]:
# GATE-1 — 비리스크 판정  (🟢 Claude → 🟠 교수 승인)
P=DEC/'removals.json'
if not P.exists():
    print('⏸  대기: 🟢 Claude가 비리스크 판정 후 생성 ->', P)
    print('   형식 {"approved": true, "removals":[{"l4_id":"RAI4-xxxx","reason":"..."}]}')
else:
    R=json.load(open(P)); rm={x['l4_id'] for x in R['removals']}
    assert R.get('approved') is True, '교수 승인 플래그(approved:true) 필요'
    assert rm <= set(by), '미존재 ID: '+str(sorted(rm-set(by))[:5])
    print('✅ 제거 승인:',len(rm),'장')

✅ 제거 승인: 40 장


In [12]:
# S3 — 제거 적용
rm={x['l4_id'] for x in json.load(open(DEC/'removals.json'))['removals']}
removed=[c for c in cards if c['l4_id'] in rm]
for c in removed:
    c['status']='retired'; c['retirement_reason']='no_harm_content'
work=[c for c in cards if c['l4_id'] not in rm]
by={c['l4_id']:c for c in work}
print(len(cards),'-',len(rm),'=',len(work))
json.dump([{'l4_id':c['l4_id'],'label_ko':c['label_ko'],'label_en':c['label_en'],
            'definition_ko':c.get('definition_ko'),'definition_en':c.get('definition_en'),
            'level':c['level'],'protected':c['protected_term']} for c in work],
          open(DEC/'_input_for_normalization.json','w'), ensure_ascii=False, indent=0)
print('-> 🟢 Claude 입력:', DEC/'_input_for_normalization.json')

1652 - 40 = 1612
-> 🟢 Claude 입력: /Users/deep1003/data3/RAI-Risk-Taxonomy/data/experiments/stage1/decisions/_input_for_normalization.json


In [13]:
# GATE-2 — 정규화 결정  (🟢 Claude)
P=DEC/'normalization.json'
if not P.exists():
    print('⏸  대기: 🟢 Claude가 정규화 결정 후 생성 ->', P)
    print('   형식 {"edits":[{"l4_id":"...","kind":"definition_form|ko_term|disambiguate|repair",')
    print('           "new_definition_ko":"...","new_definition_en":"...",')
    print('           "new_label_ko":"(선택)","new_label_en":"(선택)"}]}')
    print('   규칙: 기제·결과 동결 / 맥락 추가 금지 / 수준 변경 금지 / 보호용어 라벨 개명 금지')
else:
    E=json.load(open(P))['edits']
    print('✅ 정규화 결정:',len(E),'건',dict(Counter(e.get('kind') for e in E)))

✅ 정규화 결정: 1335 건 {'definition_form': 293, 'ko_term': 39, 'label_and_definition': 179, 'rename_and_redefine': 117, 'redefine_only': 50, 'definition_only': 222, 'label+definition': 253, 'protected_label_kept': 20, 'rename_and_definition': 93, 'adopt_prior': 52, 'definition': 17}


In [14]:
# S4 — 정규화 적용 + 보호용어 위반 / 수준 변동 자동 검사
E=json.load(open(DEC/'normalization.json'))['edits']
viol=[]
for e in E:
    c=by[e['l4_id']]
    nl=e.get('new_label_ko')
    if nl and c['protected_term'] and nl!=c['label_ko'] and not any(p in nl for p in PROTECTED):
        viol.append((c['l4_id'], c['label_ko'], nl))
assert not viol, '보호용어 개명 위반 '+str(len(viol))+'건: '+str(viol[:5])

for e in E:
    c=by[e['l4_id']]
    for k in ('label_ko','label_en','definition_ko','definition_en'):
        if e.get('new_'+k):
            c.setdefault('original_'+k, c.get(k)); c[k]=e['new_'+k]
    c['normalization']=e.get('kind')
drift=[e['l4_id'] for e in E if level(by[e['l4_id']])!=BASE_LEVEL[e['l4_id']]]
print('적용',len(E),'건 | 보호용어 위반 0 | 수준 변동',len(drift),'건',drift[:8])
if drift: print('   ※ 수준 변동은 맥락이 추가되었다는 신호 — 검토 필요')
json.dump(dict(release_id='stage1-normalized', cards=work, removed_cards=removed),
          open(OUT/'normalized_master.json','w'), ensure_ascii=False, indent=1)
print('활성:',len(work))

적용 1335 건 | 보호용어 위반 0 | 수준 변동 105 건 ['RAI4-0090', 'RAI4-0465', 'RAI4-0501', 'RAI4-0519', 'RAI4-0521', 'RAI4-0522', 'RAI4-0537', 'RAI4-0567']
   ※ 수준 변동은 맥락이 추가되었다는 신호 — 검토 필요
활성: 1612


In [15]:
# S5 — 임베딩 재생성 (텍스트 해시 캐시)
def ctext(c):
    return (c.get('label_en','') or '')+'. '+(c.get('definition_en','') or '')+' / '+(c.get('label_ko','') or '')+'. '+(c.get('definition_ko','') or '')
ids=[c['l4_id'] for c in work]; texts=[ctext(c) for c in work]
TH=hashlib.sha1(chr(30).join(texts).encode()).hexdigest()[:12]
EP=OUT/('emb_'+TH+'.npy')
if EP.exists():
    E_=np.load(EP); print('cache hit', EP.name)
else:
    from sentence_transformers import SentenceTransformer
    enc=SentenceTransformer(CFG['model']); enc.max_seq_length=CFG['max_seq_length']
    E_=enc.encode(texts, normalize_embeddings=True, batch_size=CFG['batch_size'], show_progress_bar=True).astype('float32')
    np.save(EP, E_)
EMB_SHA=hashlib.sha1(E_.tobytes()).hexdigest()[:12]
En=E_/np.linalg.norm(E_,axis=1,keepdims=True); S=En@En.T; np.fill_diagonal(S,-1)
print('E',E_.shape,'sha1',EMB_SHA)

iu=np.triu_indices(len(ids),1); v=S[iu]; ordr=np.argsort(-v)
PROT={tuple(sorted(p)) for p in CFG['cannot_link']}
pairs=[]
for k in ordr[:600]:
    i,j=int(iu[0][k]),int(iu[1][k]); a,b=ids[i],ids[j]
    pairs.append(dict(a=a,b=b,cos=round(float(v[k]),4),
        level_a=by[a]['level'], level_b=by[b]['level'], cross_level=by[a]['level']!=by[b]['level'],
        protected=tuple(sorted((a,b))) in PROT,
        label_a=by[a]['label_ko'], label_b=by[b]['label_ko'],
        label_a_en=by[a]['label_en'], label_b_en=by[b]['label_en'],
        def_a=(by[a].get('definition_ko') or '')[:220], def_b=(by[b].get('definition_ko') or '')[:220]))
json.dump(pairs, open(DEC/'_input_top_pairs.json','w'), ensure_ascii=False, indent=1)
print('-> 🟢 Claude 입력: 상위 600쌍 | cos>=0.85:',int((v>=0.85).sum()),'>=0.80:',int((v>=0.80).sum()))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/51 [00:00<?, ?it/s]

E (1612, 1024) sha1 804d16ddfcbb
-> 🟢 Claude 입력: 상위 600쌍 | cos>=0.85: 131 >=0.80: 878


In [16]:
# GATE-3 — 통합 사전  (🟢 Claude)
P=DEC/'merge_dictionary.json'
if not P.exists():
    print('⏸  대기: 🟢 Claude가 고유사 쌍 2인 판정 후 생성 ->', P)
    print('   형식 {"merges":[{"ids":[...],"survivor_id":"...","label_ko":"...","label_en":"...",')
    print('                   "definition_ko":"...","definition_en":"..."}]}')
else:
    D=json.load(open(P))['merges']
    PROT={tuple(sorted(p)) for p in CFG['cannot_link']}
    bad=[m['ids'] for m in D for p in PROT if p[0] in m['ids'] and p[1] in m['ids']]
    assert not bad, '보호 쌍 흡수: '+str(bad)
    xl=[m['ids'] for m in D if len({by[i]['level'] for i in m['ids']})>1]
    assert not xl or not CFG['block_cross_level'], '교차 수준 병합: '+str(xl[:3])
    print('✅ 사전:',len(D),'그룹,',sum(len(m['ids']) for m in D),'장')

⏸  대기: 🟢 Claude가 고유사 쌍 2인 판정 후 생성 -> /Users/deep1003/data3/RAI-Risk-Taxonomy/data/experiments/stage1/decisions/merge_dictionary.json
   형식 {"merges":[{"ids":[...],"survivor_id":"...","label_ko":"...","label_en":"...",
                   "definition_ko":"...","definition_en":"..."}]}


In [18]:
# S6 — 사전 선적용 + 제약 덴드로그램 + τ 절단 + 명명 후보 산출
D=json.load(open(DEC/'merge_dictionary.json'))['merges']
LEDGER={}; _c=[1900]
def issue(mem):
    k=tuple(sorted(mem))
    for a,b in LEDGER.items():
        if b==k: return a
    n='RAI4-'+format(_c[0],'04d'); _c[0]+=1; LEDGER[n]=k; return n
def amax(srcs,key):
    vs=[s.get(key) for s in srcs if s.get(key) is not None]
    return round(max(vs),3) if vs else None

merged=[]; used=set()
for m in sorted(D, key=lambda x: sorted(x['ids'])[0]):
    srcs=[by[i] for i in m['ids']]
    assert not (set(m['ids']) & used), '그룹 간 중복'
    used |= set(m['ids'])
    refs,seen=[],set()
    for s in srcs:
        for r in (s.get('references') or []):
            k=(r.get('title'), r.get('url'))
            if k not in seen: seen.add(k); refs.append(dict(r, source_l4_id=s['l4_id']))
    nid=issue(m['ids'])
    merged.append(dict(l4_id=nid, label_ko=m['label_ko'], label_en=m['label_en'],
        definition_ko=m['definition_ko'], definition_en=m['definition_en'],
        severity_1to5=amax(srcs,'severity_1to5'), probability_0to1=amax(srcs,'probability_0to1'),
        metrics_note='max over members; probability lower bound',
        member_metrics=[{'l4_id':s['l4_id'],'s':s.get('severity_1to5'),'p':s.get('probability_0to1')} for s in srcs],
        references=refs, status='active', stage1_source_ids=sorted(m['ids']),
        level=Counter(s['level'] for s in srcs).most_common(1)[0][0],
        protected_term=any(s['protected_term'] for s in srcs), merge_basis='dictionary'))
    for s in srcs: s['status']='retired'; s['merged_into']=nid
W=[c for c in work if c['status']=='active']+merged
byW={c['l4_id']:c for c in W}; wids=[c['l4_id'] for c in W]
print('사전 적용:',len(D),'그룹 ->',len(W),'장')

emap={ids[i]:En[i] for i in range(len(ids))}
new=[c for c in W if c['l4_id'] not in emap]
if new:
    from sentence_transformers import SentenceTransformer
    enc=SentenceTransformer(CFG['model']); enc.max_seq_length=CFG['max_seq_length']
    V=enc.encode([ctext(c) for c in new], normalize_embeddings=True, batch_size=CFG['batch_size']).astype('float32')
    for c,vv in zip(new,V): emap[c['l4_id']]=vv
EW=np.vstack([emap[i] for i in wids]).astype('float32')
EWn=EW/np.linalg.norm(EW,axis=1,keepdims=True); SW=EWn@EWn.T; np.fill_diagonal(SW,1.0)
Dm=np.clip(1-SW,0,None); np.fill_diagonal(Dm,0)
idx={k:i for i,k in enumerate(wids)}

def resolve(x):
    while x not in idx:
        nx=[k for k,vv in LEDGER.items() if x in vv]
        if not nx: raise ValueError('cannot-link '+x+' 소실')
        x=nx[0]
    return x
CL=[]
for a,b in CFG['cannot_link']:
    ra,rb=resolve(a),resolve(b)
    if ra==rb: raise ValueError('보호 쌍이 같은 병합 카드로 흡수됨: '+a+','+b)
    CL.append((ra,rb)); Dm[idx[ra],idx[rb]]=Dm[idx[rb],idx[ra]]=10.0

if CFG['block_cross_level']:
    lv=np.array([1 if byW[i]['level']=='construct' else 0 for i in wids])
    mask=lv[:,None]!=lv[None,:]
    Dm[mask]=10.0; np.fill_diagonal(Dm,0)
    print('교차 수준 차단 쌍:', int(mask.sum())//2)

Z=linkage(squareform(Dm,checks=False), method=CFG['linkage'])
S2=SW.copy(); np.fill_diagonal(S2,-1)
for a,b in CL: S2[idx[a],idx[b]]=S2[idx[b],idx[a]]=-1
tau_nm=float(S2.max())+CFG['eps']

cuts={}
for tier,tau in CFG['taus'].items():
    lab=fcluster(Z, t=1-tau, criterion='distance')
    g=defaultdict(list)
    for i,x in enumerate(lab): g[int(x)].append(wids[i])
    cuts[tier]=dict(labels=lab, groups=dict(g))
    print(tier+':', len(g), 'cards, multi', sum(1 for vv in g.values() if len(vv)>1),
          ', largest', max(len(vv) for vv in g.values()))
print('tau_nm =', format(tau_nm,'.4f'))

sweep=[(round(float(t),2), len(set(fcluster(Z,t=1-t,criterion='distance')))) for t in np.arange(0.60,0.96,0.01)]
json.dump(sweep, open(OUT/'tau_sweep.json','w'))

FP=hashlib.sha1((hashlib.sha1(EW.tobytes()).hexdigest()+json.dumps(sorted(wids))).encode()).hexdigest()[:16]
for tier in cuts:
    rows=[dict(group=g, members_key=sorted(vv), level=byW[vv[0]]['level'],
               members=[dict(l4_id=i, label_ko=byW[i]['label_ko'], label_en=byW[i]['label_en'],
                             definition_ko=(byW[i].get('definition_ko') or '')[:300],
                             protected=byW[i]['protected_term']) for i in vv])
          for g,vv in sorted(cuts[tier]['groups'].items()) if len(vv)>1]
    json.dump(dict(fingerprint=FP, tier=tier, groups=rows),
              open(DEC/('_input_naming_'+tier+'.json'),'w'), ensure_ascii=False, indent=1)
    print('-> 🟢 Claude 입력', tier+':', len(rows), '다중 군집')

사전 적용: 120 그룹 -> 1474 장


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

교차 수준 차단 쌍: 428248
C80: 1304 cards, multi 157 , largest 4
C70: 694 cards, multi 451 , largest 8
tau_nm = 0.9188
-> 🟢 Claude 입력 C80: 157 다중 군집
-> 🟢 Claude 입력 C70: 451 다중 군집


In [19]:
# GATE-4 — 명명 검정  (🟢 Claude)
missing=[t for t in ('C80','C70') if not (DEC/('naming_'+t+'.json')).exists()]
if missing:
    print('⏸  대기: 🟢 Claude가 명명 검정 후 생성 ->', ['naming_'+t+'.json' for t in missing])
    print('   fingerprint =', FP)
    print('   형식 {"fingerprint":"...","decisions":[{"group":N,"members_key":[...],')
    print('           "verdict":"approve|reject","label_ko":"...","label_en":"...",')
    print('           "definition_ko":"...","definition_en":"...","reject_reason":"기각시"}]}')
else:
    for t in ('C80','C70'):
        d=json.load(open(DEC/('naming_'+t+'.json')))
        assert d['fingerprint']==FP, t+' fingerprint 불일치 — 재검수 필요'
        a=sum(1 for x in d['decisions'] if x['verdict']=='approve')
        print('✅', t+':', len(d['decisions']), '결정 (승인', a, ', 기각', len(d['decisions'])-a, ')')

⏸  대기: 🟢 Claude가 명명 검정 후 생성 -> ['naming_C80.json', 'naming_C70.json']
   fingerprint = 53b51bd6cf7d0992
   형식 {"fingerprint":"...","decisions":[{"group":N,"members_key":[...],
           "verdict":"approve|reject","label_ko":"...","label_en":"...",
           "definition_ko":"...","definition_en":"...","reject_reason":"기각시"}]}


In [21]:
# S7 — 명명 결정 적용 → C80/C70 확정 (C70 기각의 C80 전파로 ⊇ 유지)
def load(t):
    d=json.load(open(DEC/('naming_'+t+'.json'))); assert d['fingerprint']==FP
    o={}
    for x in d['decisions']:
        g=int(x['group'])
        assert sorted(x['members_key'])==sorted(cuts[t]['groups'][g]), t+' members 불일치 g'+str(g)
        if x['verdict']=='approve':
            assert all(x.get(k) for k in ('label_ko','label_en','definition_ko','definition_en'))
        o[g]=x
    return o
d70, d80 = load('C70'), load('C80')
rej70={i for g,mem in cuts['C70']['groups'].items()
       if len(mem)>1 and d70.get(g,{}).get('verdict')!='approve' for i in mem}

def srcof(i): return sorted(set(byW[i].get('stage1_source_ids') or [i]))
tiers={}
for tier,dec in (('C80',d80),('C70',d70)):
    tc=[]; miss=0
    for g,mem in sorted(cuts[tier]['groups'].items()):
        if len(mem)==1:
            tc.append(dict(byW[mem[0]], master_source_ids=srcof(mem[0]))); continue
        x=dec.get(g)
        if x is None: miss+=1
        reject=(x is None) or (x['verdict']!='approve') or (tier=='C80' and bool(set(mem)&rej70))
        if reject:
            tc += [dict(byW[i], master_source_ids=srcof(i)) for i in mem]; continue
        srcs=[byW[i] for i in mem]
        refs,seen=[],set()
        for s in srcs:
            for r in (s.get('references') or []):
                k=(r.get('title'), r.get('url'))
                if k not in seen: seen.add(k); refs.append(dict(r, source_l4_id=r.get('source_l4_id', s['l4_id'])))
        tc.append(dict(l4_id=issue(mem), label_ko=x['label_ko'], label_en=x['label_en'],
            definition_ko=x['definition_ko'], definition_en=x['definition_en'],
            severity_1to5=amax(srcs,'severity_1to5'), probability_0to1=amax(srcs,'probability_0to1'),
            metrics_note='max over members; probability lower bound',
            member_metrics=[{'l4_id':s['l4_id'],'s':s.get('severity_1to5'),'p':s.get('probability_0to1')} for s in srcs],
            references=refs, master_source_ids=sorted({y for i in mem for y in srcof(i)}),
            level=srcs[0]['level'], merge_basis='nameability_'+tier, status='active'))
    tiers[tier]=tc
    print(tier+':', len(tc), 'cards (결정 누락', miss, ')')

C80: 1369 cards (결정 누락 0 )
C70: 937 cards (결정 누락 0 )


In [22]:
# S8 — 검증 + 산출물 + Before/After HTML
import html as H
for tier in tiers:
    lab=cuts[tier]['labels']
    assert not [(a,b) for a,b in CL if lab[idx[a]]==lab[idx[b]]], tier+' cannot-link 위반'
m70={s:c['l4_id'] for c in tiers['C70'] for s in c['master_source_ids']}
bad=[c['l4_id'] for c in tiers['C80'] if len({m70[s] for s in c['master_source_ids']})>1]
assert not bad, '중첩 위반 '+str(bad)
ref={c['l4_id'] for c in work}
for tier,tc in tiers.items():
    mp={s for c in tc for s in c['master_source_ids']}
    assert mp==ref, tier+' 매핑 불일치 누락'+str(sorted(ref-mp)[:3])+' 초과'+str(sorted(mp-ref)[:3])
    xl=[c['l4_id'] for c in tc if len(c['master_source_ids'])>1
        and len({by[s]['level'] for s in c['master_source_ids'] if s in by})>1]
    print(tier+':', len(tc), 'cards | cannot-link 0 | 매핑 무손실 | 교차 수준 병합', len(xl))

man=dict(run_at=datetime.datetime.now().isoformat(), cfg=CFG, source_sha=sha(SRC),
    embedding_sha1=EMB_SHA, fingerprint=FP, tau_nm=round(tau_nm,4),
    versions=dict(python=platform.python_version(), numpy=np.__version__, scipy=scipy.__version__),
    counts=dict(source=len(cards), after_removal=len(work), work=len(W),
                **{t:len(c) for t,c in tiers.items()}))
json.dump(man, open(OUT/'manifest.json','w'), ensure_ascii=False, indent=1, default=str)
json.dump(dict(release_id='stage1-master', cards=work, removed_cards=removed,
               id_ledger={k:list(vv) for k,vv in LEDGER.items()}),
          open(OUT/'master.json','w'), ensure_ascii=False, indent=1)
for tier,tc in tiers.items():
    json.dump(dict(release_id='stage1-'+tier, tau=CFG['taus'][tier], cards=tc),
              open(OUT/(tier.lower()+'.json'),'w'), ensure_ascii=False, indent=1)
    with open(OUT/(tier.lower()+'_mapping.csv'),'w',newline='') as f:
        wcsv=csv.writer(f); wcsv.writerow(['master_l4_id','tier_l4_id'])
        for c in tc:
            for s in c['master_source_ids']: wcsv.writerow([s, c['l4_id']])

def rows(tier):
    o=[]
    for c in sorted(tiers[tier], key=lambda x: x['l4_id']):
        s=c['master_source_ids']
        if len(s)<2: continue
        b=''.join('<div><code>'+x+'</code> '+H.escape(by[x]['label_ko'])+
                  '<div class=en>'+H.escape(by[x]['label_en'])+'</div></div>' for x in s if x in by)
        a=('<div><code>'+c['l4_id']+'</code> <b>'+H.escape(c['label_ko'])+'</b>'
           '<div class=en>'+H.escape(c['label_en'])+'</div>'
           '<div class=df>'+H.escape((c.get('definition_ko') or '')[:300])+'</div></div>')
        o.append('<tr><td class=n>'+str(len(s))+'->1</td><td>'+b+'</td><td>'+a+'</td></tr>')
    return o
CSS=("<style>body{font-family:'Apple SD Gothic Neo','Noto Sans KR',sans-serif;margin:18px}"
     "table{border-collapse:collapse;width:100%;font-size:12.5px}"
     "th,td{border:1px solid #ddd;padding:6px 8px;vertical-align:top}th{background:#f4f4f2}"
     ".n{color:#0f6e56;font-weight:600;width:56px}.en{color:#777;font-size:11.5px}"
     ".df{color:#444;font-size:11.5px;margin-top:3px}code{color:#888}h2{font-size:17px;margin-top:24px}</style>")
parts=''
for t in ('C80','C70'):
    r=rows(t)
    parts += ('<h2>'+t+' - 병합 '+str(len(r))+'건 / 전체 '+str(len(tiers[t]))+'장</h2>'
              '<table><tr><th>규모</th><th>Before</th><th>After</th></tr>'+''.join(r)+'</table>')
head=('<!doctype html><meta charset=utf-8><title>Stage-1</title>'+CSS+
      '<h1>Stage-1 결과</h1><p>Master '+str(len(work))+' · C80 '+str(len(tiers['C80']))+
      ' · C70 '+str(len(tiers['C70']))+' · tau_nm '+format(tau_nm,'.4f')+'</p>')
open(OUT/'before_after.html','w').write(head+parts)
print('완료 ->', OUT)
print(json.dumps(man['counts'], ensure_ascii=False))

C80: 1369 cards | cannot-link 0 | 매핑 무손실 | 교차 수준 병합 0
C70: 937 cards | cannot-link 0 | 매핑 무손실 | 교차 수준 병합 0
완료 -> /Users/deep1003/data3/RAI-Risk-Taxonomy/data/experiments/stage1/out
{"source": 1652, "after_removal": 1612, "work": 1474, "C80": 1369, "C70": 937}
